In [ ]:
from google.colab import files
import zipfile
import os

# Upload the 3 zip files
print("Upload your 3 zip files:")
uploaded = files.upload()

# Unzip each one
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"Unzipping {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✓ Done: {filename}")

# Check what folders were created
print("\nFolders created:")
!ls -d gemma2b*

Upload your 3 zip files:


Saving gemma2b_emotion_neutral_anger_similarity.zip to gemma2b_emotion_neutral_anger_similarity.zip
Saving gemma2b_informal_polite_similarity.zip to gemma2b_informal_polite_similarity.zip
Saving gemma2b_shakespeare_similarity.zip to gemma2b_shakespeare_similarity.zip
Unzipping gemma2b_emotion_neutral_anger_similarity.zip...
✓ Done: gemma2b_emotion_neutral_anger_similarity.zip
Unzipping gemma2b_informal_polite_similarity.zip...
✓ Done: gemma2b_informal_polite_similarity.zip
Unzipping gemma2b_shakespeare_similarity.zip...
✓ Done: gemma2b_shakespeare_similarity.zip

Folders created:
gemma2b_emotion_neutral_anger_similarity
gemma2b_emotion_neutral_anger_similarity.zip
gemma2b_informal_polite_similarity
gemma2b_informal_polite_similarity.zip
gemma2b_shakespeare_similarity
gemma2b_shakespeare_similarity.zip


In [ ]:
!pip install openai pandas

from openai import AzureOpenAI
import statistics
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
import time
import os


# Set up Azure OpenAI client
client = AzureOpenAI(
    api_key="YourKey",  # Replace with your actual key
    api_version="2024-12-01-preview",
    azure_endpoint="YourEndpoint"
)



In [ ]:
def grade_sentence(sentence: str, feature: str, n_samples: int = 3) -> float:
    prompt = (
        f"Given the following sentence, rate it from 0 to 1 based on how much "
        f"the feature '{feature}' is present, where 1 = strongly present and 0 = not at all. "
        f"Respond ONLY with a number between 0 and 1.\n\n"
        f"Sentence: {sentence}"
    )
    scores = []
    for _ in range(n_samples):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",  # or "gpt-4o-mini" - whatever model is available in your Foundry project
                messages=[
                    {"role": "system", "content": "You are an impartial linguistic rater."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=5,
                temperature=0.2
            )
            reply = response.choices[0].message.content.strip()
            value = float(reply)
            value = max(0, min(1, value))
            scores.append(value)
        except (ValueError, Exception) as e:
            print(f"Error grading sentence: {e}")
            pass

    if not scores:
        return 0.0
    return round(statistics.mean(scores), 3)


def evaluate_layer_results(layer: int, activation_type: str = "mean",
                          experiment: str = "informal_polite",
                          feature_name: str = "politeness",
                          max_sentences: int = None):
    """
    Evaluate generated sentences for a specific layer
    Modified to work with list format: [(feat_idx, feat_diff, sentences), ...]

    Args:
        layer: Which layer to evaluate (0-25)
        activation_type: "mean" or "last"
        experiment: "informal_polite", "emotion_neutral_anger", or "shakespeare"
        feature_name: What feature to grade (e.g., "politeness", "anger", "modern language")
        max_sentences: Maximum sentences to evaluate per condition (None = all sentences)
    """
    # Set paths based on experiment
    if experiment == "informal_polite":
        results_dir = "gemma2b_informal_polite_similarity"
    elif experiment == "emotion_neutral_anger":
        results_dir = "gemma2b_emotion_neutral_anger_similarity"
    elif experiment == "shakespeare":
        results_dir = "gemma2b_shakespeare_similarity"
    else:
        results_dir = experiment

    # Load the pickled results
    file_path = f"{results_dir}/similarity{layer}_{activation_type}.pkl"

    try:
        with open(file_path, "rb") as f:
            model_score_pairs = pickle.load(f)
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return None

    print(f"Evaluating Layer {layer} ({activation_type}) for {feature_name}")
    print(f"Found {len(model_score_pairs)} entries\n")

    results = []
    baseline_sentences = None

    # First pass: find baseline sentences
    for item in model_score_pairs:
        feature_idx, feature_diff, sentences = item

        if feature_idx is None:
            baseline_sentences = sentences
            print(f"Found baseline with {len(sentences)} sentences")
            break

    # Second pass: evaluate each feature
    for item in model_score_pairs:
        feature_idx, feature_diff, sentences = item

        # Skip baseline entries
        if feature_idx is None:
            continue

        print(f"\n{'='*60}")
        print(f"Feature {feature_idx} (diff: {feature_diff:.4f})")
        print(f"{'='*60}")

        feature_results = {
            'layer': layer,
            'activation_type': activation_type,
            'feature_idx': int(feature_idx),
            'feature_diff': float(feature_diff),
        }

        # Grade baseline sentences (if available)
        if baseline_sentences is not None:
            print(f"\nBASELINE:")
            # Evaluate all sentences unless max_sentences is specified
            sample_sentences = baseline_sentences[:max_sentences] if max_sentences else baseline_sentences
            baseline_scores = []

            print(f"  Grading {len(sample_sentences)} baseline sentences...")
            for i, sent in enumerate(sample_sentences, 1):
                score = grade_sentence(sent, feature_name, n_samples=3)
                baseline_scores.append(score)
                print(f"  {i}/{len(sample_sentences)}. {sent[:80]}... → {score}")
                time.sleep(0.5)

            avg_baseline = round(statistics.mean(baseline_scores), 3)
            feature_results['baseline_avg_score'] = avg_baseline
            feature_results['baseline_scores'] = baseline_scores
            feature_results['baseline_num_sentences'] = len(baseline_scores)
            print(f"\n  Average {feature_name} score: {avg_baseline}")

        # Grade steered sentences
        print(f"\nSTEERED:")
        # Evaluate all sentences unless max_sentences is specified
        sample_sentences = sentences[:max_sentences] if max_sentences else sentences
        steered_scores = []

        print(f"  Grading {len(sample_sentences)} steered sentences...")
        for i, sent in enumerate(sample_sentences, 1):
            score = grade_sentence(sent, feature_name, n_samples=3)
            steered_scores.append(score)
            print(f"  {i}/{len(sample_sentences)}. {sent[:80]}... → {score}")
            time.sleep(0.5)

        avg_steered = round(statistics.mean(steered_scores), 3)
        feature_results['steered_1_avg_score'] = avg_steered
        feature_results['steered_1_scores'] = steered_scores
        feature_results['steered_num_sentences'] = len(steered_scores)
        print(f"\n  Average {feature_name} score: {avg_steered}")

        # Calculate steering effect
        if baseline_sentences is not None:
            steering_effect = avg_steered - avg_baseline
            feature_results['steering_effect'] = round(steering_effect, 3)
            print(f"  ➜ Steering Effect: {steering_effect:+.3f}")

        results.append(feature_results)

    return results


def evaluate_all_layers(activation_type: str = "mean",
                       experiment: str = "informal_polite",
                       feature_name: str = "politeness",
                       layers_to_eval: list = None,
                       max_sentences: int = None):
    """
    Evaluate all layers and save results

    Args:
        activation_type: "mean" or "last"
        experiment: experiment name
        feature_name: what to grade
        layers_to_eval: list of layers to evaluate (None = all 26 layers)
        max_sentences: max sentences per condition (None = all)
    """
    if layers_to_eval is None:
        layers_to_eval = range(26)  # All layers

    all_results = []

    for layer in layers_to_eval:
        print(f"\n{'#'*80}")
        print(f"EVALUATING LAYER {layer}")
        print(f"{'#'*80}")

        layer_results = evaluate_layer_results(
            layer=layer,
            activation_type=activation_type,
            experiment=experiment,
            feature_name=feature_name,
            max_sentences=max_sentences
        )

        if layer_results:
            all_results.extend(layer_results)

        # Save intermediate results
        df = pd.DataFrame(all_results)
        df.to_csv(f"{experiment}_evaluation_{activation_type}.csv", index=False)
        print(f"\n✓ Saved intermediate results to CSV")

    return all_results


def compare_steering_effectiveness(results_df):
    """
    Analyze which layers and features had the most effective steering
    """
    # Calculate steering effect (steered - baseline)
    if 'steering_effect' not in results_df.columns:
        results_df['steering_effect'] = (
            results_df['steered_1_avg_score'] - results_df['baseline_avg_score']
        )

    # Sort by largest positive effect
    top_features = results_df.nlargest(10, 'steering_effect')

    print("\n" + "="*80)
    print("TOP 10 MOST EFFECTIVE STEERING FEATURES")
    print("="*80)

    for _, row in top_features.iterrows():
        print(f"\nLayer {row['layer']}, Feature {row['feature_idx']}")
        print(f"  Feature Diff: {row['feature_diff']:.4f}")
        print(f"  Baseline Score: {row['baseline_avg_score']:.3f}")
        print(f"  Steered Score: {row['steered_1_avg_score']:.3f}")
        print(f"  ➜ Steering Effect: +{row['steering_effect']:.3f}")

    return top_features


# =============================================================================
# USAGE EXAMPLES
# =============================================================================

layers_to_eval = list(range(26))

for layer in layers_to_eval:
    print(f"\n{'='*60}")
    print(f"Processing Layer {layer}...")
    print(f"{'='*60}")

    result = evaluate_all_layers(
        activation_type="last",
        experiment="informal_polite",
        feature_name="politeness",
        layers_to_eval=[layer],  # Just one layer at a time
        max_sentences=None
    )

    # Save immediately after each layer
    df = pd.DataFrame(result)
    df.to_csv(f"informal_polite_layer{layer}_last.csv", index=False)
    print(f"✓ Layer {layer} saved!")

print("\n✓ All layers done!")

Streaming output truncated to the last 5000 lines.
... → 0.2
  34/100. What hit song that was used in the Bollywood movie Don actually a reject from an... → 0
  35/100. What is a secured loan? "..\..\..\..\..\..\..\..\..\..\..\..\..\..\..\..\..\..\A... → 0
  36/100. What types of sport fish do most anglers target in New England?
Which of the fol... → 0.2
  37/100. What is stream processing ? "..\..\..\..\..\..\..\..\..\..\..\..\..\..\..\Views\... → 0
  38/100. How to get a good Survey for a product

<h2><strong>1. Get a good survey and mak... → 0.2
  39/100. How should you care for grapes? NKC03/02/2017

Grapevine care

Grapevine care in... → 0.267
  40/100. What is the difference between NumPy and pandas?

NumPy is a Python library for ... → 0.2
  41/100. Is cricket a popular sports in USA? 　　Well, I have to say that cricket is not a ... → 0.5
  42/100. How many dwarf planets are there in our solar system?cookieParser(cookieParser, ... → 0.133
  43/100. Where does the word plumbing or

In [ ]:
import pandas as pd

# List of files in the desired order
files = [
    'informal_polite_evaluation_mean.csv',
    'informal_polite_layer14.csv',
    'informal_polite_layer15.csv',
    'informal_polite_layer16.csv',
    'informal_polite_layer17.csv',
    'informal_polite_layer18.csv',
    'informal_polite_layer19.csv',
    'informal_polite_layer20.csv',
    'informal_polite_layer21.csv',
    'informal_polite_layer22.csv',
    'informal_polite_layer23.csv',
    'informal_polite_layer24.csv',
    'informal_polite_layer25.csv'
]

# Read and concatenate
dfs = [pd.read_csv(f) for f in files]
combined_df = pd.concat(dfs, ignore_index=True)

# Save the result
combined_df.to_csv('informal_polite_combined.csv', index=False)